# Comparación inicial de algoritmos con M3 y M4

Este notebook compara cuatro familias de algoritmos usando los mismos bloques de variables M3 y M4:

- Regresión logística.
- Random Forest.
- Extra Trees.
- Histogram Gradient Boosting.

La selección inicial se realiza exclusivamente con validación cruzada agrupada por conglomerado en ENDES 2024. ENDES 2025 se usa después como evaluación temporal descriptiva.

Esta etapa no busca todavía los mejores hiperparámetros. Cada algoritmo utiliza una configuración razonable y fija para identificar qué familias merecen pasar al ajuste fino. La métrica primaria es **PPV@20 %**. También se mantienen PPV@10 %, PPV@30 %, AUC, Brier y calibración.

In [ ]:
from pathlib import Path
from time import perf_counter
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
SEMILLA = 2026
N_BOOTSTRAP = 1_000

RAIZ = Path.cwd().resolve()
if RAIZ.name.lower() == "modelo":
    RAIZ = RAIZ.parent
if not (RAIZ / "datos_procesados").exists():
    raise FileNotFoundError("Ejecuta el notebook desde la raíz del proyecto o desde modelo/.")

RUTA_DATOS = RAIZ / "datos_procesados" / "tabla_modelo_2024_2025_limpia.csv"
RUTA_RESULTADOS = RAIZ / "modelo" / "resultados_comparacion_algoritmos"
print(RUTA_DATOS)

## 1. Cargar la cohorte y crear el target

`no_tomo_7d = 1` indica que el niño no tomó ninguna presentación de hierro durante los últimos siete días.

In [ ]:
datos = pd.read_csv(RUTA_DATOS, low_memory=False)
cohorte = datos.loc[
    datos["en_cohorte_modelo"].eq(True) & datos["tomo_7d"].isin([0, 1])
].copy()
cohorte["no_tomo_7d"] = (1 - cohorte["tomo_7d"]).astype("int8")

resumen = cohorte.groupby("anio")["no_tomo_7d"].agg(n="size", prevalencia="mean")
display(resumen.style.format({"prevalencia": "{:.1%}"}))

assert set(cohorte["anio"].unique()) == {2024, 2025}
assert cohorte[["anio", "caseid", "idx_nino"]].duplicated().sum() == 0

## 2. Bloques de variables

M1 y M2 se conservan como referencias. La comparación de algoritmos se realiza sobre M3 y M4. No se incluyen variables del target, razones de no consumo, identificadores ni representaciones clínicas duplicadas.

In [ ]:
PRESENTACIONES = ["recibio_jarabe", "recibio_gotas", "recibio_polvo"]
VARIABLES_M1 = ["edad_meses"]
VARIABLES_M2 = ["edad_meses", "atraso_cred", *PRESENTACIONES]
VARIABLES_M3 = [*VARIABLES_M2, "n_cred_con_fecha", "lugar_cred_codigo"]
VARIABLES_M4 = [
    *VARIABLES_M3, "anemia_oms", "edad_madre", "nivel_educativo_madre",
    "cuidador_habitual", "quintil_riqueza", "n_menores_5_hogar",
    "personas_por_dormitorio", "area",
]

BLOQUES = {"M1": VARIABLES_M1, "M2": VARIABLES_M2, "M3": VARIABLES_M3, "M4": VARIABLES_M4}
NUMERICAS = {
    "atraso_cred", "n_cred_con_fecha", "edad_madre",
    "n_menores_5_hogar", "personas_por_dormitorio"
}
BINARIAS = set(PRESENTACIONES)
CATEGORICAS = {
    "lugar_cred_codigo", "anemia_oms", "nivel_educativo_madre",
    "cuidador_habitual", "quintil_riqueza", "area"
}

ALGORITMOS = ["logistica", "random_forest", "extra_trees", "gradient_boosting"]
ESPECIFICACIONES = [
    ("M1__logistica", "M1", "logistica"),
    ("M2__logistica", "M2", "logistica"),
] + [
    (f"{bloque}__{algoritmo}", bloque, algoritmo)
    for bloque in ("M3", "M4") for algoritmo in ALGORITMOS
]

tabla_especificaciones = pd.DataFrame(ESPECIFICACIONES, columns=["modelo", "variables", "algoritmo"])
tabla_especificaciones["n_variables_originales"] = tabla_especificaciones["variables"].map(
    lambda b: len(BLOQUES[b])
)
display(tabla_especificaciones)

In [ ]:
PROHIBIDAS = {
    "no_tomo_7d", "tomo_7d", "tomo7_jarabe", "tomo7_gotas",
    "tomo7_polvo", "tomo7_otra", "tomo_estado",
    "razon_no_consumo_cod", "razon_estomago", "razon_diarrea",
    "razon_estrenimiento", "razon_sabor", "razon_dientes",
    "razon_orientacion", "razon_inicio_suplementacion",
    "razon_olvido", "razon_otra",
}
IDENTIFICADORES = {
    "anio", "caseid", "idx_nino", "hhid", "conglomerado",
    "estrato", "factor_expansion_crudo", "factor_hogar_crudo"
}
predictoras = set().union(*map(set, BLOQUES.values()))
assert not (predictoras & PROHIBIDAS)
assert not (predictoras & IDENTIFICADORES)
assert "controles_cred_acum" not in predictoras
assert len(VARIABLES_M4) == len(set(VARIABLES_M4))
print("Controles de fuga y duplicados: correctos.")

## 3. Preparación y métricas

`atraso_cred` se vuelve a calcular dentro de cada fold usando únicamente el fold de entrenamiento. PPV resuelve empates por su valor esperado para no depender del orden de las filas.

In [ ]:
def agregar_atraso_cred(entrenamiento, evaluacion):
    entrenamiento = entrenamiento.copy()
    evaluacion = evaluacion.copy()
    mediana_edad = entrenamiento.groupby("edad_meses")["controles_cred_acum"].median()
    mediana_global = entrenamiento["controles_cred_acum"].median()
    for tabla in (entrenamiento, evaluacion):
        referencia = tabla["edad_meses"].map(mediana_edad).fillna(mediana_global)
        tabla["atraso_cred"] = tabla["controles_cred_acum"] - referencia
    return entrenamiento, evaluacion


def ppv_en_fraccion(y, puntaje, fraccion):
    y = np.asarray(y, dtype=float)
    puntaje = np.asarray(puntaje, dtype=float)
    n_elegir = max(1, int(np.ceil(fraccion * len(y))))
    umbral = np.partition(puntaje, len(puntaje) - n_elegir)[len(puntaje) - n_elegir]
    superiores = puntaje > umbral
    empatados = puntaje == umbral
    faltan = n_elegir - superiores.sum()
    positivos = y[superiores].sum()
    if faltan > 0:
        positivos += faltan * y[empatados].mean()
    return float(positivos / n_elegir)


def calibracion(y, puntaje):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(puntaje, dtype=float), 1e-6, 1 - 1e-6)
    if np.std(p) < 1e-12 or len(np.unique(y)) < 2:
        return np.nan, np.nan
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    ajuste = LogisticRegression(C=1e6, max_iter=2_000).fit(logit, y)
    return float(ajuste.intercept_[0]), float(ajuste.coef_[0, 0])


def metricas(y, puntaje):
    y = np.asarray(y, dtype=int)
    puntaje = np.asarray(puntaje, dtype=float)
    intercepto, pendiente = calibracion(y, puntaje)
    return {
        "n": len(y), "prevalencia": y.mean(),
        "ppv10": ppv_en_fraccion(y, puntaje, 0.10),
        "ppv20": ppv_en_fraccion(y, puntaje, 0.20),
        "ppv30": ppv_en_fraccion(y, puntaje, 0.30),
        "auc": roc_auc_score(y, puntaje) if len(np.unique(y)) == 2 else np.nan,
        "brier": brier_score_loss(y, puntaje),
        "prediccion_media": puntaje.mean(),
        "calibracion_intercepto": intercepto,
        "calibracion_pendiente": pendiente,
    }

In [ ]:
def crear_preprocesador(variables, algoritmo):
    variables = list(variables)
    es_logistica = algoritmo == "logistica"
    transformadores = []

    if "edad_meses" in variables:
        pasos_edad = [("imputar", SimpleImputer(strategy="median"))]
        if es_logistica:
            pasos_edad.append(("spline", SplineTransformer(
                n_knots=5, degree=3, include_bias=False
            )))
        transformadores.append(("edad", Pipeline(pasos_edad), ["edad_meses"]))

    numericas = [v for v in variables if v in NUMERICAS]
    binarias = [v for v in variables if v in BINARIAS]
    categoricas = [v for v in variables if v in CATEGORICAS]

    if numericas:
        pasos_num = [("imputar", SimpleImputer(strategy="median", add_indicator=True))]
        if es_logistica:
            pasos_num.append(("escalar", StandardScaler()))
        transformadores.append(("num", Pipeline(pasos_num), numericas))
    if binarias:
        transformadores.append((
            "bin", SimpleImputer(strategy="most_frequent", add_indicator=True), binarias
        ))
    if categoricas:
        transformadores.append((
            "cat",
            Pipeline([
                ("imputar", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False
                )),
            ]),
            categoricas,
        ))
    return ColumnTransformer(transformadores, remainder="drop")


def crear_estimador(algoritmo):
    if algoritmo == "logistica":
        return LogisticRegression(max_iter=3_000)
    if algoritmo == "random_forest":
        return RandomForestClassifier(
            n_estimators=300, min_samples_leaf=20, max_features="sqrt",
            n_jobs=-1, random_state=SEMILLA
        )
    if algoritmo == "extra_trees":
        return ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=20, max_features="sqrt",
            n_jobs=-1, random_state=SEMILLA
        )
    if algoritmo == "gradient_boosting":
        return HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=200, max_leaf_nodes=15,
            min_samples_leaf=30, l2_regularization=1.0, random_state=SEMILLA
        )
    raise ValueError(f"Algoritmo desconocido: {algoritmo}")


def crear_pipeline(bloque, algoritmo):
    variables = BLOQUES[bloque]
    return Pipeline([
        ("pre", crear_preprocesador(variables, algoritmo)),
        ("modelo", crear_estimador(algoritmo)),
    ])

## 4. Comparación agrupada dentro de 2024

Todos los algoritmos reciben los mismos folds. Así, las diferencias no se deben a una partición más fácil o más difícil.

In [ ]:
desarrollo = cohorte.loc[cohorte["anio"] == 2024].reset_index(drop=True)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
particiones = list(cv.split(
    desarrollo, desarrollo["no_tomo_7d"], groups=desarrollo["conglomerado"]
))
filas_cv = []

for fold, (i_entrena, i_valida) in enumerate(particiones, start=1):
    entrena = desarrollo.iloc[i_entrena].copy()
    valida = desarrollo.iloc[i_valida].copy()
    entrena, valida = agregar_atraso_cred(entrena, valida)
    y_entrena = entrena["no_tomo_7d"].to_numpy()
    y_valida = valida["no_tomo_7d"].to_numpy()

    for nombre, bloque, algoritmo in ESPECIFICACIONES:
        variables = BLOQUES[bloque]
        modelo = crear_pipeline(bloque, algoritmo)
        inicio = perf_counter()
        modelo.fit(entrena[variables], y_entrena)
        p = modelo.predict_proba(valida[variables])[:, 1]
        segundos = perf_counter() - inicio
        filas_cv.append({
            "fold": fold, "modelo": nombre, "bloque": bloque,
            "algoritmo": algoritmo, "segundos": segundos,
            **metricas(y_valida, p),
        })
    print(f"Fold {fold}/5 completado")

resultados_cv_folds = pd.DataFrame(filas_cv)

In [ ]:
metricas_resumen = ["ppv10", "ppv20", "ppv30", "auc", "brier", "segundos"]
resultados_cv = resultados_cv_folds.groupby(["modelo", "bloque", "algoritmo"])[metricas_resumen].agg(["mean", "std"])
resultados_cv.columns = [f"{m}_{e}" for m, e in resultados_cv.columns]
resultados_cv = resultados_cv.reset_index().sort_values(
    ["ppv20_mean", "auc_mean", "brier_mean"], ascending=[False, False, True]
)
display(resultados_cv.style.format({
    c: "{:.3f}" for c in resultados_cv.columns if c.endswith(("_mean", "_std"))
}))

In [ ]:
# Los tres finalistas se eligen solo con CV 2024 y excluyen las referencias M1/M2.
ranking_candidatos = resultados_cv[resultados_cv["bloque"].isin(["M3", "M4"])].copy()
top3 = ranking_candidatos.head(3)["modelo"].tolist()
print("Top 3 según CV agrupada 2024:")
for posicion, nombre in enumerate(top3, start=1):
    fila = ranking_candidatos.loc[ranking_candidatos["modelo"] == nombre].iloc[0]
    print(f"{posicion}. {nombre}: PPV@20={fila.ppv20_mean:.1%}, AUC={fila.auc_mean:.3f}")

## 5. Evaluación temporal descriptiva 2024 → 2025

Se reajustan todas las combinaciones con todo 2024. Esta tabla muestra transporte entre años; no se utiliza para redefinir retroactivamente el top 3.

In [ ]:
entrena_2024 = cohorte.loc[cohorte["anio"] == 2024].reset_index(drop=True)
prueba_2025 = cohorte.loc[cohorte["anio"] == 2025].reset_index(drop=True)
entrena_2024, prueba_2025 = agregar_atraso_cred(entrena_2024, prueba_2025)
y_entrena = entrena_2024["no_tomo_7d"].to_numpy()
y_prueba = prueba_2025["no_tomo_7d"].to_numpy()
predicciones_2025 = {}
modelos_ajustados = {}
filas_temporales = []

for nombre, bloque, algoritmo in ESPECIFICACIONES:
    variables = BLOQUES[bloque]
    modelo = crear_pipeline(bloque, algoritmo)
    inicio = perf_counter()
    modelo.fit(entrena_2024[variables], y_entrena)
    p = modelo.predict_proba(prueba_2025[variables])[:, 1]
    segundos = perf_counter() - inicio
    modelos_ajustados[nombre] = modelo
    predicciones_2025[nombre] = p
    filas_temporales.append({
        "modelo": nombre, "bloque": bloque, "algoritmo": algoritmo,
        "segundos": segundos, **metricas(y_prueba, p),
    })

resultados_temporales = pd.DataFrame(filas_temporales).sort_values(
    ["ppv20", "auc", "brier"], ascending=[False, False, True]
)
display(resultados_temporales.style.format({
    "prevalencia": "{:.1%}", "ppv10": "{:.1%}",
    "ppv20": "{:.1%}", "ppv30": "{:.1%}",
    "auc": "{:.3f}", "brier": "{:.3f}",
    "prediccion_media": "{:.1%}", "calibracion_intercepto": "{:.3f}",
    "calibracion_pendiente": "{:.3f}", "segundos": "{:.2f}",
}))

## 6. Sensibilidad en conglomerados de 2025 no vistos en 2024

In [ ]:
conglomerados_2024 = set(entrena_2024["conglomerado"].unique())
mascara_no_vistos = ~prueba_2025["conglomerado"].isin(conglomerados_2024)
y_no_vistos = y_prueba[mascara_no_vistos.to_numpy()]
filas_no_vistos = []

for nombre, bloque, algoritmo in ESPECIFICACIONES:
    p = predicciones_2025[nombre][mascara_no_vistos.to_numpy()]
    filas_no_vistos.append({
        "modelo": nombre, "bloque": bloque, "algoritmo": algoritmo,
        **metricas(y_no_vistos, p),
    })

resultados_no_vistos = pd.DataFrame(filas_no_vistos).sort_values(
    ["ppv20", "auc"], ascending=False
)
print(f"Niños no vistos por conglomerado: {mascara_no_vistos.sum():,} de {len(prueba_2025):,} ({mascara_no_vistos.mean():.1%})")
display(resultados_no_vistos[["modelo", "ppv20", "auc", "brier", "prediccion_media"]].style.format({
    "ppv20": "{:.1%}", "auc": "{:.3f}", "brier": "{:.3f}",
    "prediccion_media": "{:.1%}",
}))

## 7. Bootstrap por conglomerados

Se compara cada finalista de CV 2024 contra M1 (solo edad). También se compara el primer finalista contra el segundo. El bootstrap se hace sobre conglomerados completos de 2025.

In [ ]:
comparaciones = [(nombre, "M1__logistica") for nombre in top3]
if len(top3) >= 2:
    comparaciones.append((top3[0], top3[1]))

rng = np.random.default_rng(SEMILLA)
grupos = prueba_2025["conglomerado"].to_numpy()
grupos_unicos = np.unique(grupos)
indices_grupo = {g: np.flatnonzero(grupos == g) for g in grupos_unicos}
filas_bootstrap = []

for modelo_a, modelo_b in comparaciones:
    diferencias_ppv = []
    diferencias_auc = []
    for _ in range(N_BOOTSTRAP):
        muestra = rng.choice(grupos_unicos, size=len(grupos_unicos), replace=True)
        indices = np.concatenate([indices_grupo[g] for g in muestra])
        y_b = y_prueba[indices]
        p_a = predicciones_2025[modelo_a][indices]
        p_b = predicciones_2025[modelo_b][indices]
        diferencias_ppv.append(
            ppv_en_fraccion(y_b, p_a, 0.20) - ppv_en_fraccion(y_b, p_b, 0.20)
        )
        diferencias_auc.append(roc_auc_score(y_b, p_a) - roc_auc_score(y_b, p_b))

    for metrica, valores in [("ppv20", diferencias_ppv), ("auc", diferencias_auc)]:
        valores = np.asarray(valores)
        filas_bootstrap.append({
            "modelo_a": modelo_a, "modelo_b": modelo_b, "metrica": metrica,
            "diferencia": (
                (ppv_en_fraccion(y_prueba, predicciones_2025[modelo_a], 0.20) -
                 ppv_en_fraccion(y_prueba, predicciones_2025[modelo_b], 0.20))
                if metrica == "ppv20" else
                (roc_auc_score(y_prueba, predicciones_2025[modelo_a]) -
                 roc_auc_score(y_prueba, predicciones_2025[modelo_b]))
            ),
            "ic95_inferior": np.percentile(valores, 2.5),
            "ic95_superior": np.percentile(valores, 97.5),
        })

resultados_bootstrap = pd.DataFrame(filas_bootstrap)
display(resultados_bootstrap.style.format({
    "diferencia": "{:+.3f}", "ic95_inferior": "{:+.3f}",
    "ic95_superior": "{:+.3f}",
}))

## 8. Equidad del primer candidato de CV 2024

Se reportan grupos con al menos 100 observaciones. Lengua materna se usa solo para auditoría y no como predictor.

In [ ]:
candidato_principal = top3[0]
auditoria = prueba_2025.copy()
auditoria["puntaje"] = predicciones_2025[candidato_principal]
filas_equidad = []

for variable in ["area", "nivel_educativo_madre", "quintil_riqueza", "lengua_materna"]:
    for grupo, sub in auditoria.groupby(variable, dropna=False):
        if len(sub) < 100:
            continue
        y_g = sub["no_tomo_7d"].to_numpy()
        p_g = sub["puntaje"].to_numpy()
        filas_equidad.append({
            "variable": variable, "grupo": str(grupo), "n": len(sub),
            "observado": y_g.mean(), "predicho": p_g.mean(),
            "ppv20": ppv_en_fraccion(y_g, p_g, 0.20),
            "auc": roc_auc_score(y_g, p_g) if len(np.unique(y_g)) == 2 else np.nan,
        })

resultados_equidad = pd.DataFrame(filas_equidad)
print(f"Candidato auditado: {candidato_principal}")
display(resultados_equidad.style.format({
    "observado": "{:.1%}", "predicho": "{:.1%}",
    "ppv20": "{:.1%}", "auc": "{:.3f}",
}))

## 9. Guardar resultados

No se exporta todavía un modelo de producción. Los tres finalistas de CV 2024 pasarán al ajuste controlado de hiperparámetros.

In [ ]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
tabla_especificaciones.to_csv(RUTA_RESULTADOS / "especificaciones.csv", index=False)
resultados_cv_folds.to_csv(RUTA_RESULTADOS / "cv_2024_folds.csv", index=False)
resultados_cv.to_csv(RUTA_RESULTADOS / "cv_2024_resumen.csv", index=False)
resultados_temporales.to_csv(RUTA_RESULTADOS / "temporal_2025.csv", index=False)
resultados_no_vistos.to_csv(RUTA_RESULTADOS / "conglomerados_no_vistos_2025.csv", index=False)
resultados_bootstrap.to_csv(RUTA_RESULTADOS / "bootstrap_2025.csv", index=False)
resultados_equidad.to_csv(RUTA_RESULTADOS / "equidad_candidato_2025.csv", index=False)
pd.DataFrame({"posicion": range(1, len(top3) + 1), "modelo": top3}).to_csv(
    RUTA_RESULTADOS / "top3_cv_2024.csv", index=False
)
print(f"Top 3: {top3}")
print(f"Resultados guardados en: {RUTA_RESULTADOS}")

## Regla para el siguiente notebook

El ajuste fino se realizará solo sobre los tres finalistas elegidos por CV agrupada de 2024. No se incorporará un algoritmo al top 3 porque haya funcionado mejor después de mirar 2025. Si un algoritmo obtiene alta PPV en 2024 pero cae mucho en 2025, se documentará como inestable.